In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.pipeline import Pipeline

from src.config import PROCESSED_DATA_DIR, FIGURES_DIR
from src.plots import save_chart
from category_encoders import TargetEncoder

vs = {"width": 1200, "height": 700, "scale": 3, "renderer": "png"}
vs_box = {"width": 1200, "height": 1200, "scale": 3, "renderer": "png"}

2025-01-13 23:12:10.997 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /Users/victor.barbarich/PycharmProjects/personal-projects/automl-itmo-insurance


In [2]:
data = pd.read_feather(PROCESSED_DATA_DIR / 'train.feather')

In [8]:
from src.plots import feature_analysis

features = ['AGE', 'GENDER', 'ANNUAL_INCOME', 'MARITAL_STATUS',
            'NUMBER_OF_DEPENDENTS', 'EDUCATION_LEVEL', 'OCCUPATION', 'HEALTH_SCORE',
            'LOCATION', 'POLICY_TYPE', 'PREVIOUS_CLAIMS', 'VEHICLE_AGE',
            'CREDIT_SCORE', 'INSURANCE_DURATION', 'CUSTOMER_FEEDBACK',
            'SMOKING_STATUS', 'EXERCISE_FREQUENCY', 'PROPERTY_TYPE']

figures = {}

vs_wide = {"width": 1200, "height": 500, "scale": 3, "renderer": "png"}
for feat in features:
    res = feature_analysis(data, feat, 'PREMIUM_AMOUNT', 'POLICY_START_DATE')
    figure = res['fig']
    # figure.show(**vs_wide)
    figures[f"{feat}_analysis".upper()] = figure


In [9]:
for name, chart in figures.items():
    save_chart(chart, name, FIGURES_DIR / 'eda' / 'features_analysis_1d', vs_wide)

In [10]:
from src.plots import crosstab

features = ['AGE', 'GENDER', 'ANNUAL_INCOME', 'MARITAL_STATUS',
            'NUMBER_OF_DEPENDENTS', 'EDUCATION_LEVEL', 'OCCUPATION', 'HEALTH_SCORE',
            'LOCATION', 'POLICY_TYPE', 'PREVIOUS_CLAIMS', 'VEHICLE_AGE',
            'CREDIT_SCORE', 'INSURANCE_DURATION', 'CUSTOMER_FEEDBACK',
            'SMOKING_STATUS', 'EXERCISE_FREQUENCY', 'PROPERTY_TYPE']

In [11]:
vs_small = {"width": 1000, "height": 500, "scale": 2, "renderer": "png"}

figures_crosstabs = {}

for i in range(len(features) - 1):
    for j in range(i + 1, len(features)):
        fig = crosstab(data, features[i], features[j], "PREMIUM_AMOUNT")
        # fig.show(**vs_small)
        figures_crosstabs[f"{features[i]}_X_{features[j]}".upper()] = fig

In [14]:
for name, chart in figures_crosstabs.items():
    save_chart(chart, name, FIGURES_DIR / 'eda' / 'features_analysis_crosstabs', vs_small)

# <b>Хорошие пересечения


<img src="reports/figures/eda/cross_tabs_static/Screenshot 2025-01-13 at 23.01.06.png" width="1024"/>

<img src="reports/figures/eda/cross_tabs_static/Screenshot 2025-01-13 at 23.04.53.png" width="1024"/>

<img src="reports/figures/eda/cross_tabs_static/Screenshot 2025-01-13 at 23.05.55.png" width="1024"/>

<img src="reports/figures/eda/cross_tabs_static/Screenshot 2025-01-13 at 23.07.42.png" width="1024"/>

<img src="reports/figures/eda/cross_tabs_static/Screenshot 2025-01-13 at 23.08.17.png" width="1024"/>


In [17]:
def combine__previous_claims_x_annual_income(data_: pd.DataFrame) -> pd.DataFrame:
    """
    Combine the previous claims and annual income features.
    If annual income more than 50k and previous claims more than 1, return 1, else 0
    """

    data = data_.copy()

    udf = lambda x: 1 if x['ANNUAL_INCOME'] > 50000 and x['PREVIOUS_CLAIMS'] > 1 else 0
    data['PREVIOUS_CLAIMS_X_ANNUAL_INCOME'] = data.apply(udf, axis=1)

    return data


def combine__education_level_x_previous_claims(data_: pd.DataFrame) -> pd.DataFrame:
    """
    Combine the education level and previous claims features.
    If previous claims more than 2 and education level is "Bachelor's" or "High School", return 1, else 0
    """

    data = data_.copy()

    udf = lambda x: 1 if x['PREVIOUS_CLAIMS'] > 2 and x['EDUCATION_LEVEL'] in ["Bachelor's", "High School"] else 0
    data['EDUCATION_LEVEL_X_PREVIOUS_CLAIMS'] = data.apply(udf, axis=1)

    return data


def check_if_none(x: str | None) -> bool:
    if x is None:
        return True
    if x == 'None':
        return True
    if pd.isna(x):
        return True
    return False


def combine__occupation_x_health_score(data_: pd.DataFrame) -> pd.DataFrame:
    """
    Combine the occupation and health score features.
    If occupation is not None but health score is None, return 1, else 0
    """

    data = data_.copy()

    udf = lambda x: 1 if (not check_if_none(x['OCCUPATION'])) and check_if_none(x['HEALTH_SCORE']) else 0
    data['OCCUPATION_X_HEALTH_SCORE'] = data.apply(udf, axis=1)

    return data


def combine__previous_claims_x_health_score(data_: pd.DataFrame) -> pd.DataFrame:
    """
    Combine the previous claims and health score features.
    If health score at least 37 and previous claims more than 1, return 1, else 0
    """

    data = data_.copy()

    udf = lambda x: 1 if x['HEALTH_SCORE'] >= 37 and x['PREVIOUS_CLAIMS'] > 1 else 0
    data['PREVIOUS_CLAIMS_X_HEALTH_SCORE'] = data.apply(udf, axis=1)

    return data


def combine__health_score_x_credit_score(data_: pd.DataFrame) -> pd.DataFrame:
    """
    Combine the health score and credit score features.
    If health score at least 37 and credit score less 550 or both are None, return 1, else 0
    """

    data = data_.copy()

    udf = lambda x: 1 if (
            (x['HEALTH_SCORE'] >= 37 and x['CREDIT_SCORE'] < 550) or
            (check_if_none(x['HEALTH_SCORE']) and check_if_none(x['CREDIT_SCORE']))
    ) else 0
    data['HEALTH_SCORE_X_CREDIT_SCORE'] = data.apply(udf, axis=1)

    return data


In [18]:
data = combine__previous_claims_x_annual_income(data)
data = combine__education_level_x_previous_claims(data)
data = combine__occupation_x_health_score(data)
data = combine__previous_claims_x_health_score(data)
data = combine__health_score_x_credit_score(data)

In [20]:
fig__previous_claims_x_annual_income = feature_analysis(data, 'PREVIOUS_CLAIMS_X_ANNUAL_INCOME', 'PREMIUM_AMOUNT', 'POLICY_START_DATE')['fig']

save_chart(fig__previous_claims_x_annual_income, 'PREVIOUS_CLAIMS_X_ANNUAL_INCOME', FIGURES_DIR / 'eda' / 'features_analysis_1d', vs_wide)
fig__previous_claims_x_annual_income.show(**vs_wide)

In [21]:
fig__education_level_x_previous_claims = feature_analysis(data, 'EDUCATION_LEVEL_X_PREVIOUS_CLAIMS', 'PREMIUM_AMOUNT', 'POLICY_START_DATE')['fig']

save_chart(fig__education_level_x_previous_claims, 'EDUCATION_LEVEL_X_PREVIOUS_CLAIMS', FIGURES_DIR / 'eda' / 'features_analysis_1d', vs_wide)
fig__education_level_x_previous_claims.show(**vs_wide)

In [22]:
fig__occupation_x_health_score = feature_analysis(data, 'OCCUPATION_X_HEALTH_SCORE', 'PREMIUM_AMOUNT', 'POLICY_START_DATE')['fig']

save_chart(fig__occupation_x_health_score, 'OCCUPATION_X_HEALTH_SCORE', FIGURES_DIR / 'eda' / 'features_analysis_1d', vs_wide)
fig__occupation_x_health_score.show(**vs_wide)

In [23]:
fig__previous_claims_x_health_score = feature_analysis(data, 'PREVIOUS_CLAIMS_X_HEALTH_SCORE', 'PREMIUM_AMOUNT', 'POLICY_START_DATE')['fig']

save_chart(fig__previous_claims_x_health_score, 'PREVIOUS_CLAIMS_X_HEALTH_SCORE', FIGURES_DIR / 'eda' / 'features_analysis_1d', vs_wide)
fig__previous_claims_x_health_score.show(**vs_wide)

In [24]:
fig__health_score_x_credit_score = feature_analysis(data, 'HEALTH_SCORE_X_CREDIT_SCORE', 'PREMIUM_AMOUNT', 'POLICY_START_DATE')['fig']

save_chart(fig__health_score_x_credit_score, 'HEALTH_SCORE_X_CREDIT_SCORE', FIGURES_DIR / 'eda' / 'features_analysis_1d', vs_wide)
fig__health_score_x_credit_score.show(**vs_wide)